In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.optim import SGD

import numpy as np

In [ ]:
class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.input_layer = nn.Linear(in_features=4, out_features=8)
        self.hidden_layer = nn.Linear(in_features=8, out_features=8)
        self.output_layer = nn.Linear(in_features=8, out_features=2)

    def forward(self, x):
        x = F.relu(self.input_layer(x))
        x = F.relu(self.hidden_layer(x))

        return self.output_layer(x)

In [ ]:
layer = nn.Linear(4, 8) # layer of 8 neurons, expecting 4 parameters as input and producing 8 as output
layer.state_dict()

In [ ]:
tens = torch.randn(1, 4) # creating random tensor of 1 row (observation) with 4 parameteres each

In [ ]:
tens

In [ ]:
layer(tens) 

In [ ]:
simple_nn = SimpleNN()

In [ ]:
simple_nn(tens)

In [ ]:
obs = 400
input_params = 4
output_params = 2
x_train = torch.randn(obs, input_params)
y_train = torch.randn(obs, output_params)

In [ ]:
simple_nn.training

In [ ]:
simple_nn.state_dict().keys()

In [ ]:
next(simple_nn.parameters())

In [ ]:
next(simple_nn.output_layer.parameters())

In [ ]:
tens

In [ ]:
next(simple_nn.parameters())

In [ ]:
simple_nn.input_layer.state_dict()

In [ ]:
simple_nn.state_dict().keys()

In [ ]:
simple_nn.state_dict()['input_layer.weight']

In [ ]:

simple_nn.input_layer.weight

In [ ]:
y_pred = simple_nn(tens)

In [ ]:
y_pred

In [ ]:
y_true = torch.tensor([[0.5, 0.5]])

In [ ]:
y_true

In [ ]:
mse_error = np.mean([y_pred.detach() - y_true]) ** 2

In [ ]:
mse_error

In [ ]:
(((-0.0892 - 0.5) + (-0.0416 - 0.5)) / 2) ** 2

In [ ]:
simple_nn.state_dict()

In [ ]:
y_true = torch.randn((obs, output_params))

In [ ]:
params = simple_nn.input_layer(tens)

In [ ]:
params.clamp(0)

In [ ]:
optim = SGD(simple_nn.parameters(), lr=3e-4)

In [ ]:
y_pred

In [109]:
class Bias:
    def __init__(self, h_size: int) -> None:
        self._h_size = h_size
        self._value = None
        
    @property
    def value(self):
        if self._value is None:
            self._value = self._generate_value()

        return self._value

    def _generate_value(self):
        return np.random.uniform(-0.5, 0.5, size=(1, self._h_size))


    def __call__(self):
        return self.value

    def __repr__(self):
        return str(self.__dict__)

class Weight:
    def __init__(self, v_size: int, h_size: int):
        self._v_size = v_size
        self._h_size = h_size
        self._value = None

    
    @property
    def value(self):
        if self._value is None:
            self._value = self._generate_value()

        return self._value

    def _generate_value(self):
        return np.random.uniform(-0.5, 0.5, size=(self._v_size, self._h_size))

    def __call__(self):
        return self.value

    def __repr__(self):
        return str(self.__dict__)
                   
class LinearLayer:
    def __init__(self, _input: int, _output: int, require_grad=True):
        self._input = _input
        self._output = _output
        self._bias = Bias(_output)()
        self._weights = Weight(_input, _output)()
        self._require_grad = require_grad
        self._last_input = None  # Store input for gradient calculation
        self._last_output = None  # Store output before activation
    
    @property
    def state_dict(self):
        return dict(
            weights = self._weights,
            bias = self._bias,
        )

    def forward(self, x, track_fn):
        """ Linear transformation + tracking """
        self._last_input = x  # Store input for backprop
        self._last_output = np.dot(x, self._weights) + self._bias  # Linear output
        track_fn.append("linear")  # Track function application in global list
        return self._last_output


    def __call__(self, track_fn, x=None):
        if x is None:
            return self.state_dict()
        else:
            return self.forward(x, track_fn)

    def __repr__(self):
        return str(self.state_dict)
        

def relu_activation(layer, track_fn):
    """ Applies ReLU activation function and tracks it """
    track_fn.append("relu")  # Track ReLU function for backprop
    return np.clip(layer, 0, np.inf)


class NeuralNetwork:

    @property
    def state_dict(self):
        return self.__dict__

    def forward(sefl, x):
        raise NotImplementedError('The forward() method must be implemented')

    def __call__(self, x):
        return self.forward(x)

class CustomNeuralNetwork(NeuralNetwork):
    def __init__(self):
        self.input_layer = LinearLayer(2, 4)
        self.hidden_layer = LinearLayer(4, 4)
        self.output_layer = LinearLayer(4, 2)

        # Store forward pass function tracking
        self._grad_fn = []

    def forward(self, x):
        x = relu_activation(self.input_layer(self._grad_fn, x), self._grad_fn)
        x = relu_activation(self.hidden_layer(self._grad_fn, x), self._grad_fn)
        x = self.output_layer(self._grad_fn, x)
        return x

    def track_functions(self):
        return self._grad_fn  # Returns list of applied functions for backprop tracking
    
    def __repr__(self):
        return self.state_dict

In [110]:
large_input = np.random.randn(1, 2)

In [111]:
nnetwork = CustomNeuralNetwork()

In [112]:
nnetwork(large_input)

array([[-0.32977128, -0.50782763]])

In [113]:
nnetwork.track_functions()

['linear', 'relu', 'linear', 'relu', 'linear']

In [76]:
layer = LinearLayer(2, 3)

In [78]:
layer.state_dict

{'weights': array([[-0.13732882,  0.08204716,  0.18751518],
        [ 0.30227301,  0.43466589,  0.47934705]]),
 'bias': array([[-0.46338898,  0.29130765, -0.41403701]])}